# Training "hey Amy"

Trimmed from openWakeWord's `automatic_model_training.ipynb` to what this wake word needs.

**Before running: Runtime → Change runtime type → T4 GPU.**

Removed from the original: the TensorFlow/tflite toolchain, which only exists to produce a `.tflite`
alongside the ONNX. The daemon loads ONNX, and those three pins (`tensorflow-cpu==2.8.1`,
`tensorflow_probability==0.16.0`, `onnx_tf==1.10.0`) date from 2022 and are the most likely thing in
the notebook to fail to install today.

Expect roughly 45–60 minutes, most of it generating speech.

## 1 · Install

In [ ]:
# Colab runs Python 3.13. piper-phonemize builds espeak-ng and its newest Linux wheel is cp311,
# so the training stack cannot be installed into this kernel at all.
#
# uv fetches a standalone 3.11 and builds a second environment beside the notebook. The downloads
# below stay in this kernel; only the three training steps run in the other one.
import sys, os
print("notebook kernel:", f"{sys.version_info.major}.{sys.version_info.minor}")

!pip install -q uv
!uv venv --python 3.11 /content/owwenv --quiet
PY = "/content/owwenv/bin/python"

# piper-sample-generator pinned to v2.0.0: master has been restructured into a package and no
# longer exposes the generate_samples module openWakeWord's trainer imports by name.
!git clone -q --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
  'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!git clone -q https://github.com/dscripka/openwakeword

# Everything the trainer needs, into the 3.11 environment. openwakeword goes in with --no-deps:
# it pins tflite-runtime and speexdsp-ns, neither of which resolves here and neither of which
# training uses. piper-phonemize-cross is the same module built for more interpreters.
# setuptools is explicit: uv builds a venv without it, and pronouncing still imports
# pkg_resources, which setuptools provides.
!uv pip install -q --python {PY} setuptools torch torchaudio numpy"<2" scipy scikit-learn pyyaml tqdm requests \
  onnxruntime webrtcvad piper-phonemize-cross deep-phonemizer==0.0.19 \
  mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 \
  audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 \
  pronouncing==0.2.0
!uv pip install -q --python {PY} --no-deps -e ./openwakeword

# The frozen feature models the classifier sits on top of.
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
base = "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1"
!wget -q {base}/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q {base}/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx

# Prove the environment before spending an hour in it.
!{PY} -c "import openwakeword, torch, pronouncing; from piper_phonemize import phonemize_espeak; \
import sys; print('training env:', '.'.join(map(str, sys.version_info[:2])), '| torch', torch.__version__, \
'| cuda', torch.cuda.is_available())"

In [ ]:
!pip install -q soundfile

import io, os, shutil, sys, glob, numpy as np, yaml, scipy.signal, scipy.io.wavfile, soundfile as sf
from pathlib import Path
from tqdm import tqdm
from huggingface_hub import snapshot_download

PY = "/content/owwenv/bin/python"   # the 3.11 interpreter the training steps run under

def save_16k_mono(src: str, dst: str) -> None:
    """
    Convert one clip to a 16 kHz mono wav.

    The corpora are fetched as files and decoded here rather than through the datasets library:
    datasets 5 routes all audio through torchcodec and raises without it, whether or not the
    column is cast, so a library meant to simplify this now stands in the way. soundfile reads
    wav and flac directly.
    """
    data, sr = sf.read(src, dtype="float32", always_2d=True)
    mono = data.mean(axis=1)
    if sr != 16000:
        mono = scipy.signal.resample_poly(mono, 16000, sr)
    scipy.io.wavfile.write(dst, 16000, (np.clip(mono, -1, 1) * 32767).astype(np.int16))

print("notebook kernel:", f"{sys.version_info.major}.{sys.version_info.minor}")

## 2 · Download the negatives

Three things: room impulse responses to make clean speech sound like a room, background noise and
music to mix under it, and precomputed embeddings of ~2000 hours of speech that the classifier
learns to say no to.

In [ ]:
# Room impulse responses (MIT survey), straight from the Hub. The repo holds 270 plain wavs
# already at 16 kHz mono, so they only need copying into place.
os.makedirs("./mit_rirs", exist_ok=True)
rir_dir = snapshot_download("davidscripka/MIT_environmental_impulse_responses",
                            repo_type="dataset", allow_patterns=["16khz/*.wav"])
for wav in tqdm(sorted(glob.glob(os.path.join(rir_dir, "16khz", "*.wav"))), desc="RIRs"):
    shutil.copy(wav, os.path.join("./mit_rirs", os.path.basename(wav)))
print(len(os.listdir("./mit_rirs")), "impulse responses")

In [ ]:
# Background audio: one AudioSet shard, a tar of flac files. This is a plain download, so nothing
# here depends on a dataset loader.
#
# The original notebook also pulled music from the rudraml/fma dataset. That repository holds only
# a loading script and no audio, and script-based datasets were removed from the library, so it
# cannot be read at all now. AudioSet's balanced shard already spans music, speech and noise, which
# is what the augmentation needs.
os.makedirs("audioset", exist_ok=True)
!wget -q --show-progress -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
!cd audioset && tar -xf bal_train09.tar

os.makedirs("./audioset_16k", exist_ok=True)
clips = sorted(Path("audioset/audio").glob("**/*.flac"))
for n, flac in enumerate(tqdm(clips, desc="background")):
    save_16k_mono(str(flac), f"./audioset_16k/bg_{n:05d}.wav")
print(len(os.listdir("./audioset_16k")), "background clips")

In [ ]:
# Precomputed speech embeddings: ~2000 hours of negatives, and an 11 hour validation set
# used to measure false alarms per hour.
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q --show-progress https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

## 3 · Configure for "hey Amy"

The phrase settings come from the config in the dsh-lite repo; the paths stay as downloaded above.

In [ ]:
import urllib.request

config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)

amy = yaml.safe_load(urllib.request.urlopen(
    "https://raw.githubusercontent.com/shaunbeach/dsh-lite/main/voice/training/hey_amy.yaml").read())

# Only the keys that describe the phrase. feature_data_files, batch_n_per_class and every path
# describe what was just downloaded and where it landed, and belong to this notebook.
for key in ("target_phrase", "model_name", "custom_negative_phrases",
            "n_samples", "n_samples_val", "steps",
            "max_negative_weight", "target_false_positives_per_hour"):
    config[key] = amy[key]

config["background_paths"] = ["./audioset_16k"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
config["output_dir"] = "./hey_amy_training"

with open("hey_amy.yaml", "w") as f:
    yaml.dump(config, f)

print(config["target_phrase"], "->", config["model_name"])
print(len(config["custom_negative_phrases"]), "custom negatives,",
      config["n_samples"], "positives,", config["steps"], "training steps")

## 4 · Generate, augment, train

The first cell is the long one. If it is interrupted, run it again: it counts what already exists
and carries on rather than starting over.

`MPLBACKEND=Agg` is set on each step because Colab exports a backend that only exists inside the
notebook kernel, and matplotlib in the training environment refuses to start with it.

In [ ]:
# Speak the phrase, and the near-misses, thousands of times
!MPLBACKEND=Agg {PY} openwakeword/openwakeword/train.py --training_config hey_amy.yaml --generate_clips

In [ ]:
# Put those clips in rooms and under noise
!MPLBACKEND=Agg {PY} openwakeword/openwakeword/train.py --training_config hey_amy.yaml --augment_clips

In [ ]:
# Train. The tflite conversion at the very end fails without TensorFlow, which is expected and
# harmless: the ONNX is written before it, and ONNX is what the daemon loads.
!MPLBACKEND=Agg {PY} openwakeword/openwakeword/train.py --training_config hey_amy.yaml --train_model || true

## 5 · Check it, then download

In [ ]:
# onnxruntime lives in the training environment, not this kernel, so the check runs there.
check = r"""
import os, sys, numpy as np, onnxruntime

path = "hey_amy_training/hey_amy.onnx"
if not os.path.exists(path):
    sys.exit("no model was produced - check the training output above")
print(f"{path}  {os.path.getsize(path)/1024:.0f} KB")

session = onnxruntime.InferenceSession(path, providers=["CPUExecutionProvider"])
name = session.get_inputs()[0].name
print("input:", name, session.get_inputs()[0].shape)

# It should load and score an embedding-shaped input. Silence ought to sit near zero; a model
# that answers high here has learned to say yes to everything.
silence = np.zeros((1, 16, 96), dtype=np.float32)
noise = np.random.randn(1, 16, 96).astype(np.float32)
print("score on silence:", float(session.run(None, {name: silence})[0][0][0]))
print("score on noise  :", float(session.run(None, {name: noise})[0][0][0]))
"""
open("check_model.py", "w").write(check)
!{PY} check_model.py

In [ ]:
from google.colab import files
files.download("hey_amy_training/hey_amy.onnx")

## On your Mac

```sh
mkdir -p ~/.dsh/wake
mv ~/Downloads/hey_amy.onnx ~/.dsh/wake/
dsh-voice --wake-word ~/.dsh/wake/hey_amy.onnx
```

It should print `Amy is listening. Say "hey amy" to wake her.`

If it fires when nobody spoke, raise `--wake-threshold` above 0.5. If it misses you, lower it and
watch the scores with `dsh-voice -v`.